In [ ]:
%pip install ipywidgets

In [ ]:
%pip install torch transformers

In [ ]:
import os
import gc
import math
import json
from datetime import datetime

import numpy as np
import pandas as pd
import librosa
import torch
import faiss

from joblib import Parallel, delayed
from sklearn.preprocessing import normalize
from transformers import ClapAudioModelWithProjection, ClapProcessor

In [ ]:
root = Path(".").resolve()
moved = 0
skipped = 0

for mp3_file in root.rglob("*.mp3"):
    if mp3_file.parent == root:
        continue

    destination = root / mp3_file.name

    if destination.exists():
        skipped += 1
        print(f"Skipping existing file: {destination.name}")
        continue

    shutil.move(str(mp3_file), str(destination))
    moved += 1

print(f"Moved: {moved}")
print(f"Skipped: {skipped}")

In [ ]:
for folder in sorted(root.rglob("*"), reverse=True):
    if folder.is_dir():
        try:
            folder.rmdir()
        except OSError:
            pass

In [ ]:
tracks = pd.read_csv("./raw_tracks.csv")
tracks

In [ ]:
audio_root = Path(".")
image_root = Path("../fma_images")
image_root.mkdir(exist_ok=True)

title_lookup = tracks["track_title"].to_dict()

def make_spectrogram(audio_path):
    track_id = int(audio_path.stem)
    track_title = title_lookup.get(track_id, audio_path.stem)
    image_path = image_root / f"{audio_path.stem}_spectrogram.png"

    if image_path.exists():
        return f"skipped {audio_path.name}"

    try:
        y, sr = librosa.load(audio_path, sr=22050, duration=30)

        S = librosa.feature.melspectrogram(
            y=y,
            sr=sr,
            n_mels=128,
            hop_length=512
        )

        S_db = librosa.power_to_db(S, ref=np.max)

        fig, ax = plt.subplots(figsize=(4.8, 2.4), dpi=100)
        fig.patch.set_facecolor("#111111")
        ax.set_facecolor("#111111")

        librosa.display.specshow(
            S_db,
            sr=sr,
            hop_length=512,
            x_axis=None,
            y_axis=None,
            cmap="magma",
            ax=ax
        )

        ax.axis("off")
        plt.tight_layout(pad=0)
        plt.savefig(
            image_path,
            dpi=100,
            facecolor=fig.get_facecolor(),
            edgecolor="none"
        )
        plt.close(fig)

        return f"done {audio_path.name} -> {track_title}"
    except Exception as e:
        return f"error {audio_path.name}: {e}"

audio_files = list(audio_root.glob("*.mp3"))

results = Parallel(
    n_jobs=max(1, os.cpu_count() - 1),
    backend="loky"
)(
    delayed(make_spectrogram)(audio_path) for audio_path in audio_files
)

In [ ]:
audio_ids = {p.stem for p in Path(".").glob("*.mp3")}
image_ids = {p.stem.replace("_spectrogram", "") for p in Path("../fma_images").glob("*_spectrogram.png")}

missing = sorted(audio_ids - image_ids)

print("Missing count:", len(missing))

missing_files = [Path(f"{track_id}.mp3") for track_id in missing]

results_missing = Parallel(
    n_jobs=max(1, os.cpu_count() - 1),
    backend="loky"
)(
    delayed(make_spectrogram)(audio_path) for audio_path in missing_files
)

for r in results_missing[:50]:
    print(r)

In [ ]:
audio_root = Path(".")

deleted = 0

for track_id in missing:
    audio_path = audio_root / f"{track_id}.mp3"

    if audio_path.exists():
        audio_path.unlink()
        deleted += 1
        print("Deleted audio:", audio_path.name)

print("Total audio deleted:", deleted)

In [ ]:
CLAP_CHECKPOINT = "laion/larger_clap_music"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

model = ClapAudioModelWithProjection.from_pretrained(CLAP_CHECKPOINT).to(device)
proc = ClapProcessor.from_pretrained(CLAP_CHECKPOINT)
model.eval()

AUDIO_DIR = "."
OUT_DIR = "../CLAP_audio_output"
os.makedirs(OUT_DIR, exist_ok=True)

TARGET_SR = 48000
FIXED_SECONDS = 30
FIXED_SAMPLES = TARGET_SR * FIXED_SECONDS

BATCH_SIZE = 64
N_JOBS = 8
SAVE_EVERY = 1000
MAX_FILES = None
TOP_K = 20

files = [f for f in os.listdir(AUDIO_DIR) if f.lower().endswith(".mp3")]
files.sort()

if MAX_FILES is not None:
    files = files[:MAX_FILES]

print("total files found:", len(files))


def load_audio_file(fname, audio_dir, target_sr, fixed_samples):
    path = os.path.join(audio_dir, fname)
    track_id = os.path.splitext(fname)[0]

    try:
        y, _ = librosa.load(path, sr=target_sr, mono=True)

        if len(y) > fixed_samples:
            y = y[:fixed_samples]
        elif len(y) < fixed_samples:
            y = np.pad(y, (0, fixed_samples - len(y)), mode="constant")

        return {
            "ok": True,
            "track_id": track_id,
            "waveform": y.astype(np.float32),
            "file": fname,
            "error": None,
        }
    except Exception as e:
        return {
            "ok": False,
            "track_id": track_id,
            "waveform": None,
            "file": fname,
            "error": str(e),
        }


track_ids = []
embeddings_parts = []
errors = []

processed_count = 0
last_saved = 0
num_batches = math.ceil(len(files) / BATCH_SIZE)

for batch_num, start in enumerate(range(0, len(files), BATCH_SIZE), start=1):
    batch_files = files[start:start + BATCH_SIZE]

    results = Parallel(n_jobs=N_JOBS, prefer="threads")(
        delayed(load_audio_file)(fname, AUDIO_DIR, TARGET_SR, FIXED_SAMPLES)
        for fname in batch_files
    )

    waveforms = []
    ids = []

    for result in results:
        if result["ok"]:
            waveforms.append(result["waveform"])
            ids.append(result["track_id"])
        else:
            errors.append((result["file"], result["error"]))

    if not waveforms:
        print(f"batch {batch_num}/{num_batches} | all files failed to load")
        continue

    try:
        inputs = proc(
            audio=waveforms,
            sampling_rate=TARGET_SR,
            return_tensors="pt",
            padding=True
        )

        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)
            batch_embeddings = outputs.audio_embeds

        batch_embeddings = batch_embeddings.detach().cpu().numpy().astype(np.float32)

        track_ids.extend(ids)
        embeddings_parts.append(batch_embeddings)
        processed_count += len(ids)

        print(
            f"batch {batch_num}/{num_batches} | "
            f"loaded {len(waveforms)} | "
            f"processed total {processed_count}"
        )

    except Exception as e:
        for fname in batch_files:
            errors.append((fname, f"batch inference failed: {e}"))
        print(f"batch {batch_num}/{num_batches} | inference failed: {e}")
        continue

    if processed_count - last_saved >= SAVE_EVERY:
        ids_arr = np.array(track_ids)
        emb_arr = np.vstack(embeddings_parts).astype(np.float32)

        np.save(os.path.join(OUT_DIR, "track_ids_checkpoint.npy"), ids_arr)
        np.save(os.path.join(OUT_DIR, "clap_embeddings_checkpoint.npy"), emb_arr)

        errors_df = pd.DataFrame(errors, columns=["file", "error"])
        errors_df.to_csv(
            os.path.join(OUT_DIR, "embedding_errors_checkpoint.csv"),
            index=False
        )

        print(f"checkpoint saved at {processed_count} songs | shape {emb_arr.shape}")
        last_saved = processed_count

    gc.collect()
    if device == "cuda":
        torch.cuda.empty_cache()

track_ids = np.array(track_ids)
embedding_dim = embeddings_parts[0].shape[1] if embeddings_parts else 512
embeddings = (
    np.vstack(embeddings_parts).astype(np.float32)
    if embeddings_parts
    else np.empty((0, embedding_dim), dtype=np.float32)
)

print("final track_ids shape:", track_ids.shape)
print("final embeddings shape:", embeddings.shape)
print("total errors:", len(errors))

np.save(os.path.join(OUT_DIR, "track_ids.npy"), track_ids)
np.save(os.path.join(OUT_DIR, "clap_embeddings.npy"), embeddings)

errors_df = pd.DataFrame(errors, columns=["file", "error"])
errors_df.to_csv(os.path.join(OUT_DIR, "embedding_errors.csv"), index=False)

print("final embedding save complete")

# normalize embeddings
embeddings_norm = normalize(embeddings, norm="l2").astype(np.float32)
np.save(os.path.join(OUT_DIR, "clap_embeddings_normalized.npy"), embeddings_norm)
print("saved normalized embeddings:", embeddings_norm.shape)

# build FAISS index
index = faiss.IndexFlatIP(embeddings_norm.shape[1])
index.add(embeddings_norm)
faiss.write_index(index, os.path.join(OUT_DIR, "clap_faiss.index"))
print("saved FAISS index")

# build neighbors csv
scores, neighbors = index.search(embeddings_norm, TOP_K + 1)

rows = []
for i, song_id in enumerate(track_ids):
    rank = 0
    for score, nbr_idx in zip(scores[i], neighbors[i]):
        neighbor_id = track_ids[nbr_idx]

        if neighbor_id == song_id:
            continue

        rank += 1
        rows.append({
            "track_id": song_id,
            "neighbor_track_id": neighbor_id,
            "rank": rank,
            "score": float(score)
        })

        if rank == TOP_K:
            break

neighbors_df = pd.DataFrame(rows)
neighbors_df.to_csv(
    os.path.join(OUT_DIR, "clap_top20_neighbors.csv"),
    index=False
)

print("saved neighbors csv")
print("rows:", len(neighbors_df))

model_info = {
    "model_name": "CLAP",
    "checkpoint": CLAP_CHECKPOINT,
    "embedding_dimension": int(embeddings_norm.shape[1]),
    "dataset": "FMA",
    "num_tracks": int(len(track_ids)),
    "neighbors_per_track": TOP_K,
    "index_type": "FAISS IndexFlatIP",
    "similarity_metric": "cosine",
    "sample_rate_hz": TARGET_SR,
    "clip_seconds": FIXED_SECONDS,
    "created": datetime.utcnow().isoformat()
}

with open(os.path.join(OUT_DIR, "clap_model_info.json"), "w") as f:
    json.dump(model_info, f, indent=2)

print("saved clap_model_info.json")

In [ ]:
OUT_DIR = "../CLAP_audio_output"

embeddings      = np.load(os.path.join(OUT_DIR, "clap_embeddings.npy")).astype("float32")
embeddings_norm = normalize(embeddings, norm="l2").astype("float32")

np.save(os.path.join(OUT_DIR, "clap_embeddings_normalized.npy"), embeddings_norm)
print("saved normalized embeddings:", embeddings_norm.shape)

track_ids       = np.load(os.path.join(OUT_DIR, "track_ids.npy"), allow_pickle=True)
embeddings_norm = np.load(os.path.join(OUT_DIR, "clap_embeddings_normalized.npy")).astype("float32")

d = embeddings_norm.shape[1]  

index = faiss.IndexFlatIP(d)
index.add(embeddings_norm)

faiss.write_index(index, os.path.join(OUT_DIR, "clap_faiss.index"))

print("index size:", index.ntotal)
print("dimension:", d)

In [ ]:
query_idx = 0
query = embeddings_norm[query_idx:query_idx+1]

scores, neighbors = index.search(query, 10)

print("query track:", track_ids[query_idx])
for score, idx in zip(scores[0], neighbors[0]):
    print(track_ids[idx], score)

In [ ]:
OUT_DIR = "../CLAP_audio_output"
TOP_K   = 20

track_ids       = np.load(os.path.join(OUT_DIR, "track_ids.npy"), allow_pickle=True)
embeddings_norm = np.load(os.path.join(OUT_DIR, "clap_embeddings_normalized.npy")).astype("float32")
index           = faiss.read_index(os.path.join(OUT_DIR, "clap_faiss.index"))

scores, neighbors = index.search(embeddings_norm, TOP_K + 1)

rows = []

for i, song_id in enumerate(track_ids):
    rank = 0
    for score, nbr_idx in zip(scores[i], neighbors[i]):
        neighbor_id = track_ids[nbr_idx]

        if neighbor_id == song_id:
            continue

        rank += 1

        rows.append({
            "track_id":          song_id,
            "neighbor_track_id": neighbor_id,
            "rank":              rank,
            "score":             float(score)
        })

        if rank == TOP_K:
            break

neighbors_df = pd.DataFrame(rows)

neighbors_df.to_csv(
    os.path.join(OUT_DIR, "clap_top20_neighbors.csv"),
    index=False
)

print("saved neighbors csv")
print("rows:", len(neighbors_df))
print("columns:", neighbors_df.columns.tolist())

In [ ]:
model_info = {
    "model_name":          "CLAP larger_clap_music",
    "checkpoint":          CLAP_CHECKPOINT,
    "embedding_dimension": int(embeddings_norm.shape[1]),
    "dataset":             "FMA",
    "num_tracks":          int(len(track_ids)),
    "neighbors_per_track": 20,
    "index_type":          "FAISS IndexFlatIP",
    "similarity_metric":   "cosine",
    "sample_rate_hz":      TARGET_SR,
    "created":             datetime.utcnow().isoformat()
}

with open("clap_model_info.json", "w") as f:
    json.dump(model_info, f, indent=2)

print("saved clap_model_info.json")

In [ ]:
track_ids    = np.load("./../CLAP_audio_output/track_ids.npy", allow_pickle=True)
index        = faiss.read_index("./../CLAP_audio_output/clap_faiss.index")
neighbors_df = pd.read_csv("./../CLAP_audio_output/clap_top20_neighbors.csv")

print("faiss vectors:",              index.ntotal)
print("track ids:",                  len(track_ids))
print("unique tracks in neighbors:", neighbors_df["track_id"].nunique())
print("max rank:",                   neighbors_df["rank"].max())
print("min rank:",                   neighbors_df["rank"].min())